<a href="https://colab.research.google.com/github/Jannflm/webtest/blob/main/testscript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import csv
import requests
import time
import json
import os

# ==========================================
# 1. CONFIGURACIÓN INICIAL
# ==========================================
URL_API = "https://aml-validation.knative.covalto.info/run/workflow"
ARCHIVO_DATOS = "pruebas_PLD.csv"
ARCHIVO_REPORTE = "reporte_qa.csv"

headers = {
    'Content-Type': 'application/json'

}

def formatea_json(valor):
    if not valor or str(valor).strip() == "":
        return "null"

    val_limpio = str(valor).strip()
    if val_limpio.endswith(".0"):
        val_limpio = val_limpio[:-2]

    return f'"{val_limpio}"'

# ==========================================
# 2. EJECUCIÓN MASIVA (CLON POSTMAN CON HARDCODE)
# ==========================================
def ejecutar_pruebas():
    resultados = []

    if not os.path.exists(ARCHIVO_DATOS):
        print(f"Error: El archivo de datos '{ARCHIVO_DATOS}' no se encontró.")
        return

    with open(ARCHIVO_DATOS, mode='r', encoding='utf-8-sig') as file:
        reader = csv.DictReader(file)

        for iteracion, row in enumerate(reader, start=1):

            # ¡AQUÍ ESTÁ EL CAMBIO!
            # La línea de economicalActivity ahora tiene el valor "8944098" fijo.
            payload_crudo = f"""{{
  "revisionMode": "N2",
  "requestId": "0",
  "channel": "App_Personas",
  "opportunityId": null,
  "idLead": null,
  "client": {{
    "clientId": "0",
    "type": "FISICA",
    "rfc": {formatea_json(row.get('rfc'))},
    "curp": {formatea_json(row.get('curp'))},
    "firstName": {formatea_json(row.get('firstName'))},
    "secondName": {formatea_json(row.get('secondName'))},
    "paternalSurname": {formatea_json(row.get('paternalSurname'))},
    "maternalSurname": {formatea_json(row.get('maternalSurname'))},
    "birthDate": {formatea_json(row.get('birthDate'))},
    "sex": "M",
    "economicalActivity": "8944098",
    "nationality": "MX",
    "addresses": [
      {{
        "addressType": "R",
        "country": "MX",
        "federalEntity": "06",
        "municipality": "002"
      }}
    ]
  }},
  "profile": {{
    "fundsSource": null,
    "fundsDestination": null,
    "transactionalAmount": null,
    "isPEP": "false"
  }}
}}"""

            inicio_tiempo = time.time()

            try:
                respuesta = requests.post(URL_API, data=payload_crudo.encode('utf-8'), headers=headers)

                tiempo_ms = round((time.time() - inicio_tiempo) * 1000)
                status_code = respuesta.status_code

                resultado_prueba = "❌ FAIL"
                detalle_error = ""
                estatus_pld = "N/A"
                detalle_coincidencia = "N/A"

                if status_code in [200, 201]:
                    resultado_prueba = "✅ PASS"

                    try:
                        resp_json = respuesta.json()

                        if isinstance(resp_json, list) and len(resp_json) > 0:
                            resp_json = resp_json[0]

                        if isinstance(resp_json, dict) and "passed" in resp_json:
                            estatus_pld = resp_json.get("passed")

                            if estatus_pld == "S":
                                detalle_coincidencia = "Sin coincidencias"
                            elif estatus_pld == "N":
                                motivos_encontrados = []
                                for step in resp_json.get("stepsExecuted", []):
                                    detalle_step = step.get("detail", "")
                                    if detalle_step and "nivel" in detalle_step.lower():
                                        motivos_encontrados.append(detalle_step)

                                if motivos_encontrados:
                                    detalle_coincidencia = " | ".join(motivos_encontrados)
                                else:
                                    razones = resp_json.get("reasons", [])
                                    detalle_coincidencia = " | ".join(razones) if razones else "Rechazado (Motivo no especificado)"
                        else:
                            estatus_pld = "RESPUESTA ASÍNCRONA"
                            detalle_coincidencia = json.dumps(resp_json, ensure_ascii=False)

                    except json.JSONDecodeError:
                        detalle_error = "Status 200 pero no era un JSON válido."
                else:
                    detalle_error = respuesta.text

                resultados.append({
                    "Iteración": iteracion,
                    "Usuario": f"{row.get('firstName', '')} {row.get('paternalSurname', '')}".strip(),
                    "Status Code": status_code,
                    "Tiempo de Respuesta (ms)": tiempo_ms,
                    "Aprobado PLD (S/N)": estatus_pld,
                    "Detalle Coincidencia": detalle_coincidencia,
                    "Resultado Petición": resultado_prueba,
                    "Detalle Error": detalle_error
                })

                print(f"Iteración {iteracion} -> PLD: {estatus_pld} | Status: {status_code}")
                time.sleep(1)

            except requests.exceptions.RequestException as e:
                pass

    # ==========================================
    # 3. GENERACIÓN DE REPORTE
    # ==========================================
    if resultados:
        campos = ["Iteración", "Usuario", "Status Code", "Tiempo de Respuesta (ms)",
                  "Aprobado PLD (S/N)", "Detalle Coincidencia", "Resultado Petición", "Detalle Error"]
        with open(ARCHIVO_REPORTE, mode='w', encoding='utf-8-sig', newline='') as file:
            writer = csv.DictWriter(file, fieldnames=campos)
            writer.writeheader()
            writer.writerows(resultados)

        print(f"\n¡Pruebas finalizadas! Se generó el reporte: {ARCHIVO_REPORTE}")

if __name__ == "__main__":
    ejecutar_pruebas()

SyntaxError: invalid decimal literal (4030124126.py, line 35)